In [0]:
# Initialize Spark session
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder.appName('Spark Playground').getOrCreate()


In [0]:

# Load the customers.csv dataset
df = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load('/Volumes/workspace/default/nandhu/C1.csv')
df.show()
df.printSchema()


+----------+-------+--------+----------+-----+--------+
|CustomerID|   Name| Country|  JoinDate|Sales|Category|
+----------+-------+--------+----------+-----+--------+
|       101|  Alice|     USA|15-01-2023|  250|       A|
|       102|    Bob|   india|01-05-2023|  150|       B|
|       103|Charlie|      UK|20-02-2023|blank|       C|
|       104|  Alice|     USA|15-01-2023|  250|       A|
|       105|    Eve|      UK|01-03-2023|  300|   Blank|
|       106|Mallory|New York|03-15-2023|  400|       B|
|       107|  Trent|   india|10-04-2023|  350|       B|
|       108|    Bob|   India|05-01-2023|  150|       B|
|       109|  Oscar|     USA|28-02-2023|  500|       A|
|       110|  Peggy|      UK|     blank|  450|       C|
+----------+-------+--------+----------+-----+--------+

root
 |-- CustomerID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- JoinDate: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Category: 

In [0]:
df=df.dropDuplicates()
df.show()

+----------+-------+--------+----------+-----+--------+
|CustomerID|   Name| Country|  JoinDate|Sales|Category|
+----------+-------+--------+----------+-----+--------+
|       106|Mallory|New York|03-15-2023|  400|       B|
|       101|  Alice|     USA|15-01-2023|  250|       A|
|       103|Charlie|      UK|20-02-2023|blank|       C|
|       110|  Peggy|      UK|     blank|  450|       C|
|       102|    Bob|   india|01-05-2023|  150|       B|
|       109|  Oscar|     USA|28-02-2023|  500|       A|
|       108|    Bob|   India|05-01-2023|  150|       B|
|       104|  Alice|     USA|15-01-2023|  250|       A|
|       105|    Eve|      UK|01-03-2023|  300|   Blank|
|       107|  Trent|   india|10-04-2023|  350|       B|
+----------+-------+--------+----------+-----+--------+



In [0]:
#to replace it with null
df=df.replace(["blank", "Blank"],None)
display(df)


CustomerID,Name,Country,JoinDate,Sales,Category
106,Mallory,New York,03-15-2023,400,B
101,Alice,USA,15-01-2023,250,A
103,Charlie,UK,20-02-2023,null,C
110,Peggy,UK,null,450,C
102,Bob,india,01-05-2023,150,B
109,Oscar,USA,28-02-2023,500,A
108,Bob,India,05-01-2023,150,B
104,Alice,USA,15-01-2023,250,A
105,Eve,UK,01-03-2023,300,null
107,Trent,india,10-04-2023,350,B


In [0]:
df=df.withColumn("Country",upper(df.Country))
display(df)

CustomerID,Name,Country,JoinDate,Sales,Category
106,Mallory,NEW YORK,03-15-2023,400,B
101,Alice,USA,15-01-2023,250,A
103,Charlie,UK,20-02-2023,null,C
110,Peggy,UK,null,450,C
102,Bob,INDIA,01-05-2023,150,B
109,Oscar,USA,28-02-2023,500,A
108,Bob,INDIA,05-01-2023,150,B
104,Alice,USA,15-01-2023,250,A
105,Eve,UK,01-03-2023,300,null
107,Trent,INDIA,10-04-2023,350,B


In [0]:
df=df.withColumn("Country",regexp_replace(df.Country,"NEW YORK","USA"))
display(df)

CustomerID,Name,Country,JoinDate,Sales,Category
106,Mallory,USA,03-15-2023,400,B
101,Alice,USA,15-01-2023,250,A
103,Charlie,UK,20-02-2023,null,C
110,Peggy,UK,null,450,C
102,Bob,INDIA,01-05-2023,150,B
109,Oscar,USA,28-02-2023,500,A
108,Bob,INDIA,05-01-2023,150,B
104,Alice,USA,15-01-2023,250,A
105,Eve,UK,01-03-2023,300,null
107,Trent,INDIA,10-04-2023,350,B


In [0]:

df=df.fillna({"Category":"Unknown","Sales":0,"JoinDate":"2023-01-01"})
display(df)

CustomerID,Name,Country,JoinDate,Sales,Category
106,Mallory,USA,03-15-2023,400,B
101,Alice,USA,15-01-2023,250,A
103,Charlie,UK,20-02-2023,0,C
110,Peggy,UK,2023-01-01,450,C
102,Bob,INDIA,01-05-2023,150,B
109,Oscar,USA,28-02-2023,500,A
108,Bob,INDIA,05-01-2023,150,B
104,Alice,USA,15-01-2023,250,A
105,Eve,UK,01-03-2023,300,Unknown
107,Trent,INDIA,10-04-2023,350,B


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, try_to_date

# Create Spark session
spark = SparkSession.builder.appName("clean_data").getOrCreate()

# Create DataFrame
data = [
    (1, 100, '2024-01-01'),
    (2, None, '2024-01-02'),
    (2, 200, 'wrong_date'),
    (3, 150, '2024-01-03'),
    (1, 100, '2024-01-01')
]

columns = ["order_id", "amount", "date"]

df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show()
# 1. Remove duplicates
df = df.dropDuplicates()
# 2. Handle null values
df = df.dropna(subset=["amount"])   # remove rows where amount is null
# 3. Fix date format
df = df.withColumn("date", try_to_date(col("date"), "yyyy-MM-dd"))
# 4. Remove invalid dates
df = df.dropna(subset=["date"])
# Final cleaned data
print("Cleaned Data:")
df.show()

Original Data:
+--------+------+----------+
|order_id|amount|      date|
+--------+------+----------+
|       1|   100|2024-01-01|
|       2|  NULL|2024-01-02|
|       2|   200|wrong_date|
|       3|   150|2024-01-03|
|       1|   100|2024-01-01|
+--------+------+----------+

Cleaned Data:
+--------+------+----------+
|order_id|amount|      date|
+--------+------+----------+
|       1|   100|2024-01-01|
|       3|   150|2024-01-03|
+--------+------+----------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, try_to_date

# Create Spark session
spark = SparkSession.builder.appName("clean_data").getOrCreate()

# Create DataFrame
data = [
    (1, 100, '2024-01-01'),
    (2, None, '2024-01-02'),
    (2, 200, 'wrong_date'),
    (3, 150, '2024-01-03'),
    (1, 100, '2024-01-01')
]

columns = ["order_id", "amount", "date"]

df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show()
# 1. Remove duplicates
df = df.dropDuplicates()
# 2. Handle null values
df = df.dropna(subset=["amount"])   # remove rows where amount is null
# 3. Fix date format
df = df.withColumn("date", try_to_date(col("date"), "yyyy-MM-dd"))
# 4. Remove invalid dates
df = df.dropna(subset=["date"])
# Final cleaned data
print("Cleaned Data:")
df.show()
     

Original Data:
+--------+------+----------+
|order_id|amount|      date|
+--------+------+----------+
|       1|   100|2024-01-01|
|       2|  NULL|2024-01-02|
|       2|   200|wrong_date|
|       3|   150|2024-01-03|
|       1|   100|2024-01-01|
+--------+------+----------+

Cleaned Data:
+--------+------+----------+
|order_id|amount|      date|
+--------+------+----------+
|       1|   100|2024-01-01|
|       3|   150|2024-01-03|
+--------+------+----------+

